# 01 — Exploratory Data Analysis

**Project:** Network Traffic Anomaly Detection (Nokia PRD)

This notebook covers PRD **Phase 2 (EDA)**: traffic distributions, class imbalance, temporal/seasonal patterns, and normal-vs-anomalous behaviour. It looks at **both tracks**:

* **Track A — NSL-KDD** (tabular intrusion records)
* **Track B — Synthetic stream** (timestamped traffic for time-series work)

> Run the Phase 1 generators first:
> ```
> python src/synthetic_stream.py
> python src/feature_engineering.py
> ```

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
RESULTS = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS, exist_ok=True)
print('Project root:', PROJECT_ROOT)

## Track A — NSL-KDD

In [ ]:
from data_loader import load_dataset, summarize, FEATURE_COLUMNS

df = load_dataset()
print(summarize(df))
df.head()

In [ ]:
# Class balance (normal vs anomaly)
counts = df['label'].value_counts().sort_index()
ax = counts.plot(kind='bar', color=['#4C9F70', '#D1495B'])
ax.set_xticklabels(['Normal (0)', 'Anomaly (1)'], rotation=0)
ax.set_title('NSL-KDD class balance')
ax.set_ylabel('count')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'eda_nslkdd_class_balance.png')); plt.show()

In [ ]:
# Distribution of a few key numeric features, normal vs anomaly
feats = ['src_bytes', 'dst_bytes', 'count', 'srv_count']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, f in zip(axes.ravel(), feats):
    for lbl, color in [(0, '#4C9F70'), (1, '#D1495B')]:
        vals = df.loc[df.label == lbl, f].clip(upper=df[f].quantile(0.99))
        ax.hist(vals, bins=40, alpha=0.5, label=f'label={lbl}', color=color)
    ax.set_title(f); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'eda_nslkdd_feature_dist.png')); plt.show()

In [ ]:
# Correlation heatmap (numeric features)
num = df[FEATURE_COLUMNS].select_dtypes(include=[np.number])
corr = num.corr()
plt.figure(figsize=(12, 10))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(); plt.title('NSL-KDD feature correlation')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'eda_nslkdd_correlation.png')); plt.show()

## Track B — Synthetic timestamped stream

In [ ]:
stream_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'stream_features.csv')
if not os.path.exists(stream_path):
    print('Run: python src/synthetic_stream.py  &&  python src/feature_engineering.py')
else:
    s = pd.read_csv(stream_path, parse_dates=['timestamp'])
    print(s.shape)
    display(s.head())

In [ ]:
# Traffic volume over time (hourly), normal vs anomaly  -> shows diurnal pattern + bursts
if os.path.exists(stream_path):
    s['hour_bin'] = s['timestamp'].dt.floor('1H')
    vol = s.groupby(['hour_bin', 'label'])['packets_count'].sum().unstack(fill_value=0)
    vol.plot(figsize=(13, 4), color={0: '#4C9F70', 1: '#D1495B'})
    plt.title('Hourly packet volume: normal vs anomaly')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'eda_stream_volume_timeline.png')); plt.show()

In [ ]:
# Average traffic by hour-of-day (seasonality the model must learn) 
if os.path.exists(stream_path):
    by_hour = s.groupby('hour_of_day')['packet_rate'].mean()
    by_hour.plot(kind='bar', figsize=(11, 4), color='#3D5A80')
    plt.title('Mean packet_rate by hour of day'); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS, 'eda_stream_hourly_profile.png')); plt.show()

In [ ]:
# Anomaly-type breakdown and how they separate in feature space
if os.path.exists(stream_path):
    print(s.loc[s.label == 1, 'anomaly_type'].value_counts())
    plt.figure(figsize=(8, 6))
    for kind, color in [('none', '#B0B0B0'), ('ddos', '#D1495B'), ('port_scan', '#E8A33D'), ('exfiltration', '#3D5A80')]:
        sub = s[s.anomaly_type == kind]
        plt.scatter(sub['packet_rate'].clip(upper=s.packet_rate.quantile(0.99)),
                    sub['bytes_per_packet'].clip(upper=s.bytes_per_packet.quantile(0.99)),
                    s=6, alpha=0.4, label=kind, color=color)
    plt.xlabel('packet_rate'); plt.ylabel('bytes_per_packet'); plt.legend()
    plt.title('Feature-space separation by anomaly type')
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'eda_stream_anomaly_scatter.png')); plt.show()

## Takeaways for modelling

* **Imbalance** (~95/5) confirmed → use recall/precision/PR-AUC, not accuracy (Phase 3).
* **Diurnal + weekend seasonality** is present in the stream → motivates temporal features and the LSTM (Phase 2).
* **Anomaly types separate** along `packet_rate` / `bytes_per_packet` → these engineered features are valuable inputs.
* DDoS/scan bursts are **clustered in time** → streaming detection with a rolling baseline is appropriate (Phase 4).